In [0]:
# IMPORT 
from pyspark.sql import functions as f
from delta.tables import DeltaTable
import re
from datetime import datetime

In [0]:
# VARIABLES CONFIGURATION
AUDIT_TABLE = "`urban-mobility-data-platform-dev`.bronze.ingestion_audit"
BRONZE_TABLE = "`urban-mobility-data-platform-dev`.bronze.yellow_trips_raw"
SOURCE_PATH = "/Volumes/urban-mobility-data-platform-dev/source/data/"

RECORD_COLUMNS =  [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee"
]

In [0]:
# DISCOVER ALL YELLOW partquet taxi files
all_files = dbutils.fs.ls(SOURCE_PATH)
yellow_files = sorted([file.path  for file in all_files if re.match(r"yellow_tripdata_\d{4}-\d{2}\.parquet$", file.name)])
print("Yellow Files Discovered: ", len(yellow_files))
for file_path in yellow_files:
    print(file_path)

In [0]:
# Find Files that already have processing successful
audit_df = spark.table(AUDIT_TABLE)
processed_files = {
    row["source_file"]
    for row in audit_df
    .filter(f.col("status") == "SUCCESS")
    .select("source_file")
    .collect()
}
print("Successfully Processed Files: ", len(processed_files))
for x in sorted(processed_files):
    print(x)

In [0]:
# Detect New Files to Process
new_files = [ file_path  
             for file_path in yellow_files 
             if file_path not in processed_files
            ]
print("New Files Detected: ", len(new_files))
for file_path in new_files:
    print(f"NEW -> {file_path}")

In [0]:
# Process Each New File
bronze_table = DeltaTable.forName(spark, BRONZE_TABLE)
ingestion_batch_id = (
    f"batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
)
for file_path in new_files:
    file_name = file_path.split("/")[-1]
    month_match = re.search(r"(\d{4}-\d{2})", file_name)
    if not month_match:
        raise ValueError(f"Could not determine Source month from {file_name}")
    source_month = month_match.group(1)
    print(f"Processing.. -> {file_name}")
    print(f"Month.. -> {source_month}")

    # Read Source Parquet File
    df = spark.read.parquet(file_path)

    df = (
        df
        .withColumn(
            "_record_hash",
            f.sha2(
                f.to_json(
                    f.struct(
                        *[f.col(c) for c in RECORD_COLUMNS]
                    )
                ),
                256
            )
        )
        .withColumn("_source_file", f.lit(file_path))
        .withColumn("_source_month", f.lit(source_month))
        .withColumn("_ingested_at", f.current_timestamp())
        .withColumn("_ingestion_batch_id", f.lit(ingestion_batch_id))
    )
      
    # Remove exact hash duplictes from this incoming data from parquet
    df = df.dropDuplicates(["_record_hash"])

    source_record_count = df.count()
    print(f"Unique Records in this File: {source_record_count}")

    # Merge records into bronze
    (
        bronze_table.alias("target")
        .merge(
            df.alias("source"),
            "target._record_hash = source._record_hash"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Bronze Merge Completed for File name: {file_name}")
    # Mark File as successfully processed
    audit_record = spark.createDataFrame(
        [
            (
                file_path,
                source_month,
                "SUCCESS",
                source_record_count
            )
        ],
        [
            "source_file",
            "source_month",
            "status",
            "record_count"
        ]
    ).withColumn(
        "processed_at",
        f.current_timestamp()
    )
    audit_record.write.mode("append").saveAsTable(AUDIT_TABLE)

    print("Audit Updated for: ", file_name)


In [0]:
print("\n========== INGESTION SUMMARY ==========")
print(f"Files discovered : {len(yellow_files)}")
print(f"Files processed  : {len(new_files)}")
print(f"Files skipped    : {len(yellow_files) - len(new_files)}")
print("=======================================")